# 06 · Pair-Correlation-Style Comparison: Zeta vs Poisson vs GUE Matrix

This notebook moves from nearest-neighbor spacing statistics to a first **pair-correlation-style** comparison.

Earlier notebooks studied:

- raw spacings
- unfolded spacings
- ratio statistics
- finite GUE matrix simulations

Here we look at a broader two-point quantity built from many pairwise level differences.

## Why this matters

A central result in the zeta-zero story is that, after unfolding, the two-point statistics of the zeros are expected to resemble **GUE** behavior rather than **Poisson** behavior.

This notebook is still exploratory:

- zeta zeros come from `mpmath`
- unfolding is done with a transparent local-density approximation
- GUE is modeled numerically using finite random Hermitian matrices
- we compute an empirical pair-correlation-style histogram rather than a fully normalized closed-form object

So this is a practical lab notebook, not a theorem notebook.

In [ ]:
import numpy as np
import mpmath as mp
import matplotlib.pyplot as plt

mp.mp.dps = 50
rng = np.random.default_rng(9423)

## Zeta zeros and raw gaps

In [ ]:
N = 450
zeros = [mp.zetazero(n) for n in range(1, N + 1)]
t = np.array([float(mp.im(z)) for z in zeros])
zeta_gaps = np.diff(t)

len(t), len(zeta_gaps), t[:5]

## Local unfolding helper

To compare pairwise spacings on a roughly stationary scale, we first build an unfolded coordinate by dividing each local gap by a moving local mean and then cumulatively summing.

This is a lightweight numerical unfolding step.

In [ ]:
def local_normalize_gaps(gaps, window=15):
    kernel = np.ones(window) / window
    local_means = np.convolve(gaps, kernel, mode="same")
    half = window // 2
    for i in range(half):
        local_means[i] = gaps[:i + half + 1].mean()
        local_means[-(i + 1)] = gaps[-(i + half + 1):].mean()
    return gaps / local_means

def unfold_from_gaps(gaps, window=15):
    unfolded_gaps = local_normalize_gaps(gaps, window=window)
    x = np.concatenate([[0.0], np.cumsum(unfolded_gaps)])
    return x, unfolded_gaps

In [ ]:
zeta_x, zeta_unfolded_gaps = unfold_from_gaps(zeta_gaps, window=15)
zeta_unfolded_gaps.mean(), zeta_unfolded_gaps.std()

## Poisson control

In [ ]:
poisson_gaps = rng.exponential(scale=1.0, size=len(zeta_gaps))
poisson_x, poisson_unfolded_gaps = unfold_from_gaps(poisson_gaps, window=15)

poisson_unfolded_gaps.mean(), poisson_unfolded_gaps.std()

## Finite GUE matrix reference

We generate random complex Hermitian matrices, extract bulk eigenvalue gaps, and unfold that bulk spacing sequence.

In [ ]:
def sample_gue_bulk_spacings(matrix_size=80, n_matrices=24, bulk_fraction=0.5, rng=None):
    if rng is None:
        rng = np.random.default_rng()

    all_gaps = []

    for _ in range(n_matrices):
        real = rng.normal(size=(matrix_size, matrix_size))
        imag = rng.normal(size=(matrix_size, matrix_size))
        A = real + 1j * imag
        H = (A + A.conj().T) / 2.0

        eigvals = np.linalg.eigvalsh(H)
        eigvals = np.sort(eigvals)

        n = len(eigvals)
        keep = int(n * bulk_fraction)
        start = (n - keep) // 2
        stop = start + keep

        bulk_vals = eigvals[start:stop]
        bulk_gaps = np.diff(bulk_vals)
        all_gaps.extend(bulk_gaps.tolist())

    return np.array(all_gaps)

gue_gaps = sample_gue_bulk_spacings(matrix_size=80, n_matrices=24, bulk_fraction=0.5, rng=rng)
gue_x, gue_unfolded_gaps = unfold_from_gaps(gue_gaps, window=15)

gue_unfolded_gaps.mean(), gue_unfolded_gaps.std(), len(gue_x)

## Quick spacing sanity check

Before computing pair statistics, compare the unfolded nearest-neighbor gaps.

In [ ]:
plt.figure(figsize=(8.8, 5.0))
bins = np.linspace(0, 3.5, 30)
plt.hist(zeta_unfolded_gaps, bins=bins, density=True, alpha=0.55, label="zeta")
plt.hist(poisson_unfolded_gaps, bins=bins, density=True, alpha=0.45, label="Poisson")
plt.hist(gue_unfolded_gaps, bins=bins, density=True, alpha=0.45, label="GUE matrix")
plt.xlabel("unfolded spacing")
plt.ylabel("density")
plt.title("Unfolded nearest-neighbor spacing check")
plt.legend()
plt.show()

## Empirical pair-correlation-style histogram

For an unfolded sequence \(x_n\), we compute differences

\[
\Delta_{m,n} = x_m - x_n
\]

for pairs with \(m>n\), and collect positive differences in a finite range.

This is a practical histogram-based approximation to a two-point spacing comparison.
It is not the fully normalized analytic pair-correlation function, but it is enough to compare:

- Poisson-like behavior
- GUE-like behavior
- zeta behavior

In [ ]:
def pair_differences(x, max_sep=6.0, max_index_offset=60):
    diffs = []
    n = len(x)
    for i in range(n):
        jmax = min(n, i + 1 + max_index_offset)
        d = x[i+1:jmax] - x[i]
        d = d[(d > 0) & (d <= max_sep)]
        if len(d):
            diffs.extend(d.tolist())
    return np.array(diffs)

zeta_pair = pair_differences(zeta_x, max_sep=6.0, max_index_offset=60)
poisson_pair = pair_differences(poisson_x, max_sep=6.0, max_index_offset=60)
gue_pair = pair_differences(gue_x, max_sep=6.0, max_index_offset=60)

len(zeta_pair), len(poisson_pair), len(gue_pair)

## Pair-difference histogram

The visual distinction to look for:

- **Poisson** should allow more near-coincidences
- **GUE / zeta** should show stronger local repulsion

In [ ]:
plt.figure(figsize=(8.8, 5.2))
bins = np.linspace(0, 6.0, 40)
plt.hist(zeta_pair, bins=bins, density=True, alpha=0.55, label="zeta")
plt.hist(poisson_pair, bins=bins, density=True, alpha=0.45, label="Poisson")
plt.hist(gue_pair, bins=bins, density=True, alpha=0.45, label="GUE matrix")
plt.xlabel("pair difference")
plt.ylabel("density")
plt.title("Pair-correlation-style comparison")
plt.legend()
plt.show()

## Small-separation region

This is the most sensitive region for repulsion.

In [ ]:
small_bins = np.linspace(0, 2.0, 28)

plt.figure(figsize=(8.8, 5.0))
plt.hist(zeta_pair, bins=small_bins, density=True, alpha=0.55, label="zeta")
plt.hist(poisson_pair, bins=small_bins, density=True, alpha=0.45, label="Poisson")
plt.hist(gue_pair, bins=small_bins, density=True, alpha=0.45, label="GUE matrix")
plt.xlabel("pair difference")
plt.ylabel("density")
plt.title("Small-separation pair comparison")
plt.legend()
plt.show()

## Compare with a standard GUE-style reference curve

A canonical GUE pair-correlation expression is

\[
1 - \left(\frac{\sin \pi s}{\pi s}\right)^2.
\]

We overlay this only as a **reference shape**.
Our empirical histogram is not normalized to coincide exactly with that analytic object, so the comparison here is qualitative.

In [ ]:
s = np.linspace(0.001, 6.0, 600)
gue_pair_reference = 1 - (np.sin(np.pi * s) / (np.pi * s))**2

plt.figure(figsize=(8.8, 5.2))
plt.hist(zeta_pair, bins=np.linspace(0, 6.0, 40), density=True, alpha=0.40, label="zeta pair histogram")
plt.hist(gue_pair, bins=np.linspace(0, 6.0, 40), density=True, alpha=0.30, label="GUE matrix pair histogram")
plt.plot(s, gue_pair_reference / np.trapz(gue_pair_reference, s), linewidth=2, label="normalized GUE reference shape")
plt.xlabel("pair difference")
plt.ylabel("density")
plt.title("Pair histogram with GUE-style reference shape")
plt.legend()
plt.show()

## Running local pair-density estimate

For each point \(x_n\), count how many later points fall within a short window.
This gives a simple local two-point density diagnostic.

In [ ]:
def local_pair_count(x, radius=1.0, max_index_offset=60):
    counts = []
    n = len(x)
    for i in range(n):
        jmax = min(n, i + 1 + max_index_offset)
        d = x[i+1:jmax] - x[i]
        counts.append(np.sum((d > 0) & (d <= radius)))
    return np.array(counts)

zeta_counts = local_pair_count(zeta_x, radius=1.0, max_index_offset=60)
poisson_counts = local_pair_count(poisson_x, radius=1.0, max_index_offset=60)
gue_counts = local_pair_count(gue_x, radius=1.0, max_index_offset=60)

zeta_counts.mean(), poisson_counts.mean(), gue_counts.mean()

In [ ]:
plt.figure(figsize=(8.8, 5.0))
bins = np.arange(0, max(zeta_counts.max(), poisson_counts.max(), gue_counts.max()) + 2) - 0.5
plt.hist(zeta_counts, bins=bins, density=True, alpha=0.55, label="zeta")
plt.hist(poisson_counts, bins=bins, density=True, alpha=0.45, label="Poisson")
plt.hist(gue_counts, bins=bins, density=True, alpha=0.45, label="GUE matrix")
plt.xlabel("count of later points within radius 1")
plt.ylabel("density")
plt.title("Local pair-count comparison")
plt.legend()
plt.show()

## Quantitative summaries

In [ ]:
def summary_stats(x):
    return {
        "mean": float(np.mean(x)),
        "std": float(np.std(x)),
        "min": float(np.min(x)),
        "max": float(np.max(x)),
    }

summary = {
    "zeta_unfolded_gaps": summary_stats(zeta_unfolded_gaps),
    "poisson_unfolded_gaps": summary_stats(poisson_unfolded_gaps),
    "gue_unfolded_gaps": summary_stats(gue_unfolded_gaps),
    "zeta_pair_diffs": summary_stats(zeta_pair),
    "poisson_pair_diffs": summary_stats(poisson_pair),
    "gue_pair_diffs": summary_stats(gue_pair),
    "zeta_local_pair_counts": summary_stats(zeta_counts),
    "poisson_local_pair_counts": summary_stats(poisson_counts),
    "gue_local_pair_counts": summary_stats(gue_counts),
}
summary

## Notes

- This notebook computes a **pair-correlation-style histogram**, not a fully normalized analytic pair-correlation function.
- The main practical comparison is whether zeta looks closer to the finite GUE matrix data than to the Poisson control.
- Small-separation behavior is the most informative region for local repulsion.
- The analytic shape
  \[
  1 - \left(\frac{\sin \pi s}{\pi s}\right)^2
  \]
  is included only as a qualitative reference curve.

## Next directions

A natural next notebook could do one of these:

1. refine the normalization to get closer to a standard pair-correlation estimator  
2. increase matrix size and sample count for the GUE reference  
3. compare higher-order spacing correlations  
4. build 4-gap and 5-gap local descriptors as a less standard extension